In [7]:
##### רועי אשכנזי #####
##### 206775991 #####
##### https://github.com/RoidoAsh/Movie_Project_Part_1 ######

In [2]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import urllib.request
import time

In [4]:
# הגדרת רשימת הקבצים להורדה מ-IMDb
datasets = {
    "title.basics.tsv.gz": "https://datasets.imdbws.com/title.basics.tsv.gz",
    "title.ratings.tsv.gz": "https://datasets.imdbws.com/title.ratings.tsv.gz",
    "title.principals.tsv.gz": "https://datasets.imdbws.com/title.principals.tsv.gz",
    "name.basics.tsv.gz": "https://datasets.imdbws.com/name.basics.tsv.gz"
}

print("Starting download of IMDb datasets...")

for filename, url in datasets.items():
    urllib.request.urlretrieve(url, filename)
    print(f"Successfully downloaded: {filename}")

print("All datasets are ready.")

Starting download of IMDb datasets...
Successfully downloaded: title.basics.tsv.gz
Successfully downloaded: title.ratings.tsv.gz
Successfully downloaded: title.principals.tsv.gz
Successfully downloaded: name.basics.tsv.gz
All datasets are ready.


In [6]:
#  טעינה וסינון נתונים 
df_basics = pd.read_csv('title.basics.tsv.gz', sep='\t', compression='gzip', low_memory=False)

# המרה למספרים וטיפול בערכים חסרים
for col in ['startYear', 'runtimeMinutes']:
    df_basics[col] = pd.to_numeric(df_basics[col].replace('\\N', np.nan), errors='coerce')

# סינון: סרטים, שנים עד 2024, משך 60-300 דקות
df_movies = df_basics[
    (df_basics['titleType'] == 'movie') & 
    (df_basics['startYear'] <= 2024) & 
    (df_basics['runtimeMinutes'].between(60, 300))
].dropna(subset=['startYear', 'runtimeMinutes']).copy()

print(f"Movies after filtering: {len(df_movies)}")

# --- חיבור הדירוגים ---
df_ratings = pd.read_csv('title.ratings.tsv.gz', sep='\t', compression='gzip')
final_df = pd.merge(df_movies, df_ratings, on='tconst', how='inner')

print(f"Movies after merging ratings: {len(final_df)}")

display(final_df[['tconst', 'primaryTitle', 'averageRating']].head())

Movies after filtering: 388976
Movies after merging ratings: 280932


,tconst,primaryTitle,averageRating
0,tt0000147,The Corbett-Fitzsimmons Fight,5.3
1,tt0000502,Bohemios,3.5
2,tt0000574,The Story of the Kelly Gang,6.0
3,tt0000591,The Prodigal Son,4.8
4,tt0000679,The Fairylogue and Radio-Plays,4.9


In [12]:
# הסבר על מה עשיתי עד עכשיו 
# 388715 זה מספר הסרטים שמצאתי בתוך מיליוני הסרטים שעומדים בסינון של בין שעה לחמש שעות
# 280610 זה מספר הסרטים שעומדים גם בסינון של בין שעה לחמש שעות וגם בסינון של דירוג גולשים 

In [ ]:
# בתא הבא אני בעצם מחלץ את חמשת השחקנים המובילים

In [7]:
# יצירת סט למניעת חיפושים איטיים
relevant_movies = set(final_df['tconst'])

actors_list = []
reader = pd.read_csv('title.principals.tsv.gz', sep='\t', compression='gzip', chunksize=500000)

for chunk in reader:
    # סינון שורות רלוונטיות
    filtered = chunk[
        (chunk['tconst'].isin(relevant_movies)) & 
        (chunk['category'].isin(['actor', 'actress']))
    ]
    if not filtered.empty:
        actors_list.append(filtered[['tconst', 'nconst', 'ordering']])

# איחוד וסידור הנתונים
all_actors = pd.concat(actors_list)

# חילוץ 5 השחקנים הראשונים לפי סדר הופעה (ordering)
lead_actors = (all_actors.sort_values(['tconst', 'ordering'])
               .groupby('tconst')['nconst']
               .apply(lambda x: list(x)[:5])
               .reset_index(name='lead_actors_ids'))

# חיבור לטבלה המרכזית
final_df = pd.merge(final_df, lead_actors, on='tconst', how='left')

display(final_df[['primaryTitle', 'lead_actors_ids']].head())

,primaryTitle,lead_actors_ids
0,The Corbett-Fitzsimmons Fight,NaN
1,Bohemios,"[nm0215752, nm0252720]"
2,The Story of the Kelly Gang,"[nm0846887, nm0846894, nm1431224, nm3002376, n..."
3,The Prodigal Son,"[nm0906197, nm0332182, nm1323543, nm1759558]"
4,The Fairylogue and Radio-Plays,"[nm0000875, nm0122665, nm0122665, nm0933446, n..."


In [ ]:
# בשורה הראשונה יש ערך חסר בשדה של השחקנים וזאת מכיוון שהסרט מאוד ישן וכנראה שאין תיעוד של השחקנים בתקופה ההיא 

In [8]:
# הצגת סיכום ה-Dataset לאחר איחוד הנתונים
print(f"מבנה הטבלה: {final_df.shape[0]} רשומות, {final_df.shape[1]} עמודות.")
display(final_df.head(10))

מבנה הטבלה: 280932 רשומות, 12 עמודות.


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids
0,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897.0,\N,100.0,"Documentary,News,Sport",5.3,603,NaN
1,tt0000502,movie,Bohemios,Bohemios,0,1905.0,\N,100.0,\N,3.5,26,"[nm0215752, nm0252720]"
2,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,\N,70.0,"Action,Adventure,Biography",6.0,1069,"[nm0846887, nm0846894, nm1431224, nm3002376, n..."
3,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907.0,\N,90.0,Drama,4.8,40,"[nm0906197, nm0332182, nm1323543, nm1759558]"
4,tt0000679,movie,The Fairylogue and Radio-Plays,The Fairylogue and Radio-Plays,0,1908.0,\N,120.0,"Adventure,Fantasy",4.9,86,"[nm0000875, nm0122665, nm0122665, nm0933446, n..."
5,tt0001756,movie,Lucha por la herencia,Lucha por la herencia,0,1911.0,\N,92.0,\N,4.8,9,"[nm0034454, nm0034548, nm0679170, nm0688764, n..."
6,tt0001790,movie,"Les Misérables, Part 1: Jean Valjean",Les misérables - Époque 1: Jean Valjean,0,1913.0,\N,60.0,Drama,5.9,69,"[nm0470307, nm0959921, nm0893346, nm0592965, n..."
7,tt0002026,movie,Anny - Story of a Prostitute,Anny - en gatepiges roman,0,1912.0,\N,68.0,"Drama,Romance",4.6,19,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
8,tt0002101,movie,Cleopatra,Cleopatra,0,1912.0,\N,100.0,"Drama,History",5.1,673,"[nm0306947, nm0801774, nm0276160, nm0733482, n..."
9,tt0002130,movie,Dante's Inferno,L'inferno,0,1911.0,\N,71.0,"Adventure,Drama,Fantasy",7.1,4120,"[nm0660139, nm0685283, nm0209738, nm0209738, n..."


In [ ]:
# הקוד הבא ינקה לנו את השורות בהן אין שחקנים

In [12]:
# סינון שורות ללא שחקנים ראשיים
df_final = final_df.dropna(subset=['lead_actors_ids']).copy()

# בחירת 5,000 הסרטים עם מספר ההצבעות הגבוה ביותר (הפופולריים ביותר)
df_sample = df_final.sort_values(by='numVotes', ascending=False).head(5000).copy()

# הכנת עמודה לעלילה (תתמלא בהמשך)
df_sample['plot'] = ""

# תצוגת דוגמה מהנתונים לאחר הסינון
display(df_sample[['tconst', 'primaryTitle', 'numVotes', 'lead_actors_ids']].head())

,tconst,primaryTitle,numVotes,lead_actors_ids
57551,tt0111161,The Shawshank Redemption,3185031,"[nm0000209, nm0000151, nm0348409, nm0006669, n..."
121629,tt0468569,The Dark Knight,3164142,"[nm0000288, nm0005132, nm0001173, nm0000323, n..."
157748,tt1375666,Inception,2813777,"[nm0000138, nm0330687, nm0680983, nm0913822, n..."
66640,tt0137523,Fight Club,2604651,"[nm0000093, nm0001570, nm0001533, nm0340260, n..."
126670,tt0816692,Interstellar,2524464,"[nm0000190, nm0004266, nm1567113, nm3237775, n..."


In [ ]:
# שלב הבא שלי יהיה למשוך את התקציר באמצעות המפתח 

In [ ]:
# להריץ את הקוד הבא לפני שאני מריץ את התא של הלולאה שממשיכה למשוך תקציר לעוד 1000 סרטים 

In [3]:
# טעינת המאסטר 
df_sample = pd.read_csv('Master_Movies_Dataset.csv')

In [3]:
# API Key OMDb - a8e3a79f
# 1. טעינת קובץ המאסטר
df_sample = pd.read_csv('Master_Movies_Dataset.csv')

# הגדרות ה-API
API_KEY = "a8e3a79f"

print("מתחיל בתהליך משיכת התקצירים לקובץ המאסטר...")
processed_count = 0

for index, row in df_sample.iterrows():
    # בדיקה: מדלג רק אם יש תקציר תקין (לא NaN, לא N/A ולא Error)
    plot_val = df_sample.at[index, 'plot']
    
    if pd.notna(plot_val) and plot_val != 'N/A' and plot_val != 'Error':
        continue
        
    # בניית הכתובת לבקשה
    movie_id = row['tconst']
    url = f"http://www.omdbapi.com/?i={movie_id}&apikey={API_KEY}"
    
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        
        # שמירת התקציר אם נמצא, אחרת נרשום שלא נמצא
        if data.get('Response') == 'True':
            df_sample.at[index, 'plot'] = data.get('Plot', 'N/A')
        else:
            df_sample.at[index, 'plot'] = 'N/A'
            
    except Exception as e:
        df_sample.at[index, 'plot'] = 'Error'
    
    # השהיה קלה למניעת חסימה מהשרת
    time.sleep(0.3)
    
    # עדכון המונה והדפסת התקדמות כל 100 סרטים
    processed_count += 1
    if processed_count % 100 == 0:
        print(f"הושלמה העבודה על {processed_count} סרטים נוספים...")
        # שמירה לקובץ המאסטר
        df_sample.to_csv('Master_Movies_Dataset.csv', index=False)

# שמירה סופית לאחר סיום הלולאה
df_sample.to_csv('Master_Movies_Dataset.csv', index=False)
print("סיום העבודה! הנתונים נשמרו בקובץ המאסטר.")

# תצוגה סופית
display(df_sample[['primaryTitle', 'plot']].tail())

מתחיל בתהליך משיכת התקצירים לקובץ המאסטר...
הושלמה העבודה על 100 סרטים נוספים...
הושלמה העבודה על 200 סרטים נוספים...
הושלמה העבודה על 300 סרטים נוספים...
הושלמה העבודה על 400 סרטים נוספים...
הושלמה העבודה על 500 סרטים נוספים...
הושלמה העבודה על 600 סרטים נוספים...
הושלמה העבודה על 700 סרטים נוספים...
הושלמה העבודה על 800 סרטים נוספים...
הושלמה העבודה על 900 סרטים נוספים...
הושלמה העבודה על 1000 סרטים נוספים...
סיום העבודה! הנתונים נשמרו בקובץ המאסטר.


,primaryTitle,plot
4995,The Shack,"A grieving man receives a mysterious, personal..."
4996,Exorcist: The Beginning,"In 1947, having abandoned his faith, Father Me..."
4997,We Are Your Friends,An aspiring disc jockey falls for his mentor's...
4998,License to Wed,A reverend puts an engaged couple through a gr...
4999,Bulletproof,"Two criminals, Keats and Moses, end their frie..."


In [4]:
# בדיקה כמה סרטים מלאים עם תקציר
filled_plots = df_sample[df_sample['plot'].notna() & (df_sample['plot'] != 'N/A') & (df_sample['plot'] != 'Error')]
print(f"יש לי כבר {len(filled_plots)} סרטים עם תקציר מלא.")

יש לי כבר 5000 סרטים עם תקציר מלא.


In [ ]:
# עכשיו אאסוף מידע על תקציב והכנסות בוויקפדיה באמצעות Web Scraping 

In [30]:
# הגדרת כותרות כדי להיראות כמו דפדפן אמיתי
# הגדרות כלליות
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

def get_wiki_financials(title):
    """מחלץ ומנקה נתונים פיננסיים מוויקיפדיה בצורה עמידה יותר"""
    # החלפת רווחים בקו תחתון לפורמט URL
    title_formatted = title.replace(' ', '_')
    urls = [f"https://en.wikipedia.org/wiki/{title_formatted}_(film)",
            f"https://en.wikipedia.org/wiki/{title_formatted}"]
    
    data = {'budget': None, 'box_office': None}
    
    for url in urls:
        try:
            response = requests.get(url, headers=HEADERS, timeout=5)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                infobox = soup.find('table', {'class': 'infobox'})
                
                if infobox:
                    for row in infobox.find_all('tr'):
                        header = row.find('th')
                        cell = row.find('td')
                        if header and cell:
                            label = header.get_text().lower()
                            # ניקוי הערות שוליים ורווחים
                            cleaned_text = re.sub(r'\[.*?\]', '', cell.get_text()).strip()
                            
                            if 'budget' in label and data['budget'] is None:
                                data['budget'] = cleaned_text
                            elif 'box office' in label and data['box_office'] is None:
                                data['box_office'] = cleaned_text
                    
                    # אם מצאנו לפחות משהו, נחזיר אותו
                    if data['budget'] or data['box_office']:
                        return data
        except Exception as e:
            continue
            
    return data

# --- תהליך העבודה ---
print("טוען את קובץ המאסטר...")
df_sample = pd.read_csv('Master_Movies_Dataset.csv')

# הכרחת העמודות להיות מסוג Object (טקסט) למניעת שגיאות
df_sample['budget'] = df_sample['budget'].astype(object)
df_sample['box_office'] = df_sample['box_office'].astype(object)

print("מתחיל תהליך משיכת נתונים פיננסיים...")

for idx, row in df_sample.iterrows():
    # בדיקת דילוג חכמה: מדלג אם יש כבר נתונים תקינים ב-budget
    # בודק אם הערך אינו None, אינו N/A, אינו מחרוזת ריקה, ואינו NaN
    val = row.get('budget')
    if pd.notna(val) and str(val).lower() not in ['n/a', 'none', '']:
        continue
        
    # משיכת הנתונים
    info = get_wiki_financials(row['primaryTitle'])
    
    # עדכון רק אם מצאנו מידע חדש
    if info['budget']:
        df_sample.at[idx, 'budget'] = info['budget']
    if info['box_office']:
        df_sample.at[idx, 'box_office'] = info['box_office']
    
    # השהיה למניעת חסימה
    time.sleep(0.5)
    
    # דיווח ושמירה כל 10 סרטים
    if (idx + 1) % 10 == 0:
        print(f"הושלמו {idx + 1} סרטים...")
        df_sample.to_csv('Master_Movies_Dataset.csv', index=False)

# שמירה סופית
df_sample.to_csv('Master_Movies_Dataset.csv', index=False)
print("סיום משיכת הנתונים! הקובץ נשמר.")

# הצגת טבלה מסכמת בסיום
display(df_sample[['primaryTitle', 'budget', 'box_office']].dropna(subset=['budget']).head(15))

טוען את קובץ המאסטר...
מתחיל תהליך משיכת נתונים פיננסיים...
הושלמו 160 סרטים...
הושלמו 170 סרטים...
הושלמו 250 סרטים...
הושלמו 430 סרטים...
הושלמו 470 סרטים...
הושלמו 560 סרטים...
הושלמו 630 סרטים...
הושלמו 650 סרטים...
הושלמו 670 סרטים...
הושלמו 690 סרטים...
הושלמו 860 סרטים...
הושלמו 880 סרטים...
הושלמו 970 סרטים...
הושלמו 980 סרטים...
הושלמו 1000 סרטים...
הושלמו 1040 סרטים...
הושלמו 1060 סרטים...
הושלמו 1150 סרטים...
הושלמו 1210 סרטים...
הושלמו 1280 סרטים...
הושלמו 1290 סרטים...
הושלמו 1320 סרטים...
הושלמו 1340 סרטים...
הושלמו 1350 סרטים...
הושלמו 1360 סרטים...
הושלמו 1370 סרטים...
הושלמו 1380 סרטים...
הושלמו 1390 סרטים...
הושלמו 1400 סרטים...
הושלמו 1410 סרטים...
הושלמו 1420 סרטים...
הושלמו 1430 סרטים...
הושלמו 1440 סרטים...
הושלמו 1450 סרטים...
הושלמו 1460 סרטים...
הושלמו 1470 סרטים...
הושלמו 1480 סרטים...
הושלמו 1490 סרטים...
הושלמו 1500 סרטים...
הושלמו 1510 סרטים...
הושלמו 1520 סרטים...
הושלמו 1530 סרטים...
הושלמו 1540 סרטים...
הושלמו 1550 סרטים...
הושלמו 1560 סרטים...
הושלמו 15

הושלמו 5000 סרטים...
סיום משיכת הנתונים! הקובץ נשמר.


,primaryTitle,budget,box_office
0,The Shawshank Redemption,$25 million,$73.3 million
1,The Dark Knight,$185 million,$1.009 billion
2,Inception,$160 million,$839 million
3,Fight Club,$63–65 million,$102 million
4,Interstellar,$165 million,$773.8 million
5,Forrest Gump,$55 million,$678 million
6,Pulp Fiction,$8–8.5 million,$213.9 million
7,The Matrix,$63 million,$467.8 million
8,The Godfather,$6–7 million,$250–291 million
9,The Lord of the Rings: The Fellowship of the Ring,$93 million,$897 million


In [5]:
# הגדרת המשתנים שבודקים למי יש ערך (לא ריק) וכמה סרטים יש עם נתונים פיננסים
filled_budget = df_sample['budget'].notna()
filled_box_office = df_sample['box_office'].notna()

print(f"יש לי כבר {filled_budget.sum()} סרטים עם נתוני תקציב (Budget).")
print(f"יש לי כבר {filled_box_office.sum()} סרטים עם נתוני הכנסות (Box Office).")

יש לי כבר 3444 סרטים עם נתוני תקציב (Budget).
יש לי כבר 3533 סרטים עם נתוני הכנסות (Box Office).


In [ ]:
# עכשיו אאסוף מידע על מדינה ושפה בוויקפדיה באמצעות Web Scraping 

In [32]:
# פונקציית מטא-דאטה ממוקדת
def get_wiki_metadata(title):
    title_formatted = title.replace(' ', '_')
    urls = [f"https://en.wikipedia.org/wiki/{title_formatted}_(film)",
            f"https://en.wikipedia.org/wiki/{title_formatted}"]
    
    data = {'language': None, 'country': None}
    
    for url in urls:
        try:
            response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0...'}, timeout=5)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                infobox = soup.find('table', {'class': 'infobox'})
                if infobox:
                    for row in infobox.find_all('tr'):
                        header = row.find('th')
                        cell = row.find('td')
                        if header and cell:
                            label = header.get_text().lower()
                            if 'language' in label and data['language'] is None:
                                data['language'] = cell.get_text().strip()
                            elif 'country' in label and data['country'] is None:
                                data['country'] = cell.get_text().strip()
                    if data['language'] or data['country']: return data
        except Exception: continue
    return data

# טעינת המאסטר המעודכן
df_sample = pd.read_csv('Master_Movies_Dataset.csv')

# לולאת השלמה - רצה על הכל ומחפשת רק את מה שחסר בעמודות החדשות
print("מתחיל משיכת שפה ומדינה...")
for idx, row in df_sample.iterrows():
    # אם כבר יש שפה ומדינה - דלג!
    if pd.notna(row.get('language')) and pd.notna(row.get('country')):
        continue
        
    info = get_wiki_metadata(row['primaryTitle'])
    if info['language']: df_sample.at[idx, 'language'] = info['language']
    if info['country']: df_sample.at[idx, 'country'] = info['country']
    
    time.sleep(0.3)
    if (idx + 1) % 50 == 0:
        print(f"הושלמו {idx + 1} סרטים בסבב מטא-דאטה...")
        df_sample.to_csv('Master_Movies_Dataset.csv', index=False)

df_sample.to_csv('Master_Movies_Dataset.csv', index=False)
print("סיום איסוף שפה ומדינה!")

מתחיל משיכת שפה ומדינה...
הושלמו 50 סרטים בסבב מטא-דאטה...
הושלמו 100 סרטים בסבב מטא-דאטה...
הושלמו 150 סרטים בסבב מטא-דאטה...
הושלמו 200 סרטים בסבב מטא-דאטה...
הושלמו 250 סרטים בסבב מטא-דאטה...
הושלמו 300 סרטים בסבב מטא-דאטה...
הושלמו 350 סרטים בסבב מטא-דאטה...
הושלמו 400 סרטים בסבב מטא-דאטה...
הושלמו 450 סרטים בסבב מטא-דאטה...
הושלמו 500 סרטים בסבב מטא-דאטה...
הושלמו 550 סרטים בסבב מטא-דאטה...
הושלמו 600 סרטים בסבב מטא-דאטה...
הושלמו 650 סרטים בסבב מטא-דאטה...
הושלמו 700 סרטים בסבב מטא-דאטה...
הושלמו 750 סרטים בסבב מטא-דאטה...
הושלמו 800 סרטים בסבב מטא-דאטה...
הושלמו 850 סרטים בסבב מטא-דאטה...
הושלמו 900 סרטים בסבב מטא-דאטה...
הושלמו 950 סרטים בסבב מטא-דאטה...
הושלמו 1000 סרטים בסבב מטא-דאטה...
הושלמו 1050 סרטים בסבב מטא-דאטה...
הושלמו 1100 סרטים בסבב מטא-דאטה...
הושלמו 1150 סרטים בסבב מטא-דאטה...
הושלמו 1200 סרטים בסבב מטא-דאטה...
הושלמו 1250 סרטים בסבב מטא-דאטה...
הושלמו 1300 סרטים בסבב מטא-דאטה...
הושלמו 1350 סרטים בסבב מטא-דאטה...
הושלמו 1400 סרטים בסבב מטא-דאטה...
הושלמו 1450 סר

In [10]:
# בודק את שלמות הטבלה והנתונים הקיימים
df_check = pd.read_csv('Master_Movies_Dataset.csv')

print(f"סה\"כ שורות בקובץ המאסטר: {len(df_check)}") 

# בודק כמה סרטים יש עליהם מידע קיים על מדינה ושפה
filled_countries = df_check['country'].notna().sum()
filled_languages = df_check['language'].notna().sum()

print(f"מספר הסרטים שיש עליהם מידע על מדינה: {filled_countries}")
print(f"מספר הסרטים שיש עליהם מידע על שפה: {filled_languages}")

סה"כ שורות בקובץ המאסטר: 5000
מספר הסרטים שיש עליהם מידע על מדינה: 2862
מספר הסרטים שיש עליהם מידע על שפה: 3844


In [ ]:
#מנקה את העמודה endYear  

In [4]:
# 2. בדיקה ומחיקה של העמודה endYear
if 'endYear' in df_sample.columns:
    df_sample = df_sample.drop(columns=['endYear'])
    print(" הצלחה: העמודה 'endYear' נמחקה מזיכרון הריצה!")
else:
    print(" שים לב: העמודה 'endYear' לא נמצאה")

# 3. שמירה פיזית מחדש של הקובץ הדרוס (דריסה בטוחה)
df_sample.to_csv('Master_Movies_Dataset.csv', index=False)

# 4. בדיקת אימות - הצגת העמודות שנותרו בבסיס הנתונים
print(list(df_sample.columns))

 שים לב: העמודה 'endYear' לא נמצאה
['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'lead_actors_ids', 'plot', 'budget', 'box_office', 'language', 'country']


In [ ]:
# בדיקת ערכים חסרים

In [10]:
df = pd.read_csv('Master_Movies_Dataset.csv')
total = len(df)
for col in df.columns:
    missing = df[col].isna().sum()
    pct = (missing / total) * 100
    print(f"עמודה {col}: {missing} ערכים חסרים ({pct:.2f}%)")

עמודה tconst: 0 ערכים חסרים (0.00%)
עמודה titleType: 0 ערכים חסרים (0.00%)
עמודה primaryTitle: 0 ערכים חסרים (0.00%)
עמודה originalTitle: 0 ערכים חסרים (0.00%)
עמודה isAdult: 0 ערכים חסרים (0.00%)
עמודה startYear: 0 ערכים חסרים (0.00%)
עמודה runtimeMinutes: 0 ערכים חסרים (0.00%)
עמודה genres: 0 ערכים חסרים (0.00%)
עמודה averageRating: 0 ערכים חסרים (0.00%)
עמודה numVotes: 0 ערכים חסרים (0.00%)
עמודה lead_actors_ids: 0 ערכים חסרים (0.00%)
עמודה plot: 0 ערכים חסרים (0.00%)
עמודה budget: 1556 ערכים חסרים (31.12%)
עמודה box_office: 1467 ערכים חסרים (29.34%)
עמודה language: 1156 ערכים חסרים (23.12%)
עמודה country: 2138 ערכים חסרים (42.76%)
